# Broadcast Communication

By default, communication occurs between all satellites that are specified by the communication
method. This tutorial shows how to use two classes to configure a broadcast action that must be
taken for communication to occur:

* [Broadcast](../api_reference/act/index.html#bsk_rl.act.Broadcast), which gives satellites an action
  that enables communication from them at the end of the current step.
* [BroadcastCommunication](../api_reference/comm/index.html#bsk_rl.comm.BroadcastCommunication), which
  can be combined with another communication method to limit communication from broadcasters to
  those satellites satisfying the requirements of the other communication method.

## Configuring the Environment

For this example, a multisatellite target imaging environment will be used. The goal is
to maximize the value of unique images taken. This configuration is similar to the 
[Multi-Agent Environments](../examples/multiagent_envs.html) example.

In [1]:
from bsk_rl import sats, act, obs, scene, data, comm
from bsk_rl.sim import dyn, fsw


class ImagingSatellite(sats.ImagingSatellite):
    observation_spec = [
        obs.OpportunityProperties(
            dict(prop="priority"),
            dict(prop="opportunity_open", norm=5700.0),
            n_ahead_observe=4,
        )
    ]
    action_spec = [act.Broadcast(duration=15.0), act.Image(n_ahead_image=4)]
    dyn_type = dyn.FullFeaturedDynModel
    fsw_type = fsw.SteeringImagerFSWModel


ACTION_BROADCAST = 0
ACTION_IMAGE_0 = 1
ACTION_IMAGE_1 = 2
ACTION_IMAGE_2 = 3
ACTION_IMAGE_3 = 4

Satellite properties are set to give the satellite near-unlimited power and storage resources. To randomize some parameters in a correlated manner across satellites, a ``sat_arg_randomizer`` is set and passed to the environment. In this case, the satellites are distributed in a trivial single-plane Walker-delta constellation.

In [2]:
from bsk_rl.utils.orbital import walker_delta_args

N_AGENTS = 2
sat_args = dict(
    imageAttErrorRequirement=0.01,
    imageRateErrorRequirement=0.01,
    batteryStorageCapacity=1e9,
    storedCharge_Init=1e9,
    dataStorageCapacity=1e12,
    u_max=0.4,
    K1=0.25,
    K3=3.0,
    omega_max=0.087,
    servo_Ki=5.0,
    servo_P=150 / 5,
)
sat_arg_randomizer = walker_delta_args(
    altitude=800.0, inc=60.0, n_planes=1, clustersize=N_AGENTS, clusterspacing=5.0
)

A communication type is defined that uses multidegree line-of-sight communication with broadcasting.

In [3]:
class BroadcastLOS(comm.BroadcastCommunication, comm.LOSMultiCommunication):
    pass

Finally, the environment can be instantiated.

In [4]:
from bsk_rl import ConstellationTasking

env = ConstellationTasking(
    satellites=[ImagingSatellite(f"EO-{i + 1}", sat_args) for i in range(N_AGENTS)],
    scenario=scene.UniformTargets(1000),
    rewarder=data.UniqueImageReward(),
    communicator=BroadcastLOS(),
    sat_arg_randomizer=sat_arg_randomizer,
    log_level="INFO",
)
_ = env.reset()

2026-05-19 20:28:45,930 gym                            INFO       Resetting environment with seed=3933954310


2026-05-19 20:28:45,932 scene.targets                  INFO       Generating 1000 targets


2026-05-19 20:28:45,997 sats.satellite.EO-1            INFO       <0.00> EO-1: Finding opportunity windows from 0.00 to 600.00 seconds


2026-05-19 20:28:46,029 sats.satellite.EO-2            INFO       <0.00> EO-2: Finding opportunity windows from 0.00 to 600.00 seconds


2026-05-19 20:28:46,059 gym                            INFO       <0.00> Environment reset


In the first step, both agents are tasked with an imaging action, and one successfully images a target.

In [5]:
_ = env.step({"EO-1": ACTION_IMAGE_3, "EO-2": ACTION_IMAGE_2})

2026-05-19 20:28:46,064 gym                            INFO       <0.00> === STARTING STEP ===


2026-05-19 20:28:46,065 sats.satellite.EO-1            INFO       <0.00> EO-1: target index 3 tasked


2026-05-19 20:28:46,065 sats.satellite.EO-1            INFO       <0.00> EO-1: Target(tgt-150) tasked for imaging


2026-05-19 20:28:46,066 sats.satellite.EO-1            INFO       <0.00> EO-1: Target(tgt-150) window enabled: 487.4 to 600.0


2026-05-19 20:28:46,067 sats.satellite.EO-1            INFO       <0.00> EO-1: setting timed terminal event at 600.0


...

2026-05-19 20:28:46,070 sats.satellite.EO-2            INFO       <0.00> EO-2: setting timed terminal event at 193.4


2026-05-19 20:28:46,091 sats.satellite.EO-2            INFO       <110.00> EO-2: imaged Target(tgt-642)


2026-05-19 20:28:46,092 data.base                      INFO       <110.00> Total reward: {'EO-2': 0.020813305978741536}


2026-05-19 20:28:46,093 sats.satellite.EO-2            INFO       <110.00> EO-2: Satellite EO-2 requires retasking


2026-05-19 20:28:46,095 gym                            INFO       <110.00> Step reward: {'EO-2': 0.020813305978741536}


When both select the broadcast action, data is shared in both directions. This is subject to line-of-sight availability as well as selecting the correct action.

In [6]:
_ = env.step({"EO-1": ACTION_BROADCAST, "EO-2": ACTION_BROADCAST})

2026-05-19 20:28:46,100 gym                            INFO       <110.00> === STARTING STEP ===


2026-05-19 20:28:46,101 sats.satellite.EO-1            INFO       <110.00> EO-1: setting timed terminal event at 125.0


2026-05-19 20:28:46,101 sats.satellite.EO-2            INFO       <110.00> EO-2: setting timed terminal event at 125.0


2026-05-19 20:28:46,106 sats.satellite.EO-1            INFO       <125.00> EO-1: timed termination at 125.0 for broadcast


2026-05-19 20:28:46,106 sats.satellite.EO-2            INFO       <125.00> EO-2: timed termination at 125.0 for broadcast


...

2026-05-19 20:28:46,108 comm.communication             INFO       <125.00> Communicating data in 2 directions.


2026-05-19 20:28:46,108 comm.communication             INFO       <125.00> Optimizing data communication between all pairs of satellites


2026-05-19 20:28:46,109 sats.satellite.EO-1            INFO       <125.00> EO-1: Satellite EO-1 requires retasking


2026-05-19 20:28:46,109 sats.satellite.EO-2            INFO       <125.00> EO-2: Satellite EO-2 requires retasking


2026-05-19 20:28:46,112 gym                            INFO       <125.00> Step reward: {}


One agent can broadcast while the others are completing a different task.

In [7]:
_ = env.step({"EO-1": ACTION_IMAGE_2, "EO-2": ACTION_BROADCAST})  # Agent 1 broadcasts

2026-05-19 20:28:46,117 gym                            INFO       <125.00> === STARTING STEP ===


2026-05-19 20:28:46,118 sats.satellite.EO-1            INFO       <125.00> EO-1: target index 2 tasked


2026-05-19 20:28:46,118 sats.satellite.EO-1            INFO       <125.00> EO-1: Target(tgt-396) tasked for imaging


2026-05-19 20:28:46,119 sats.satellite.EO-1            INFO       <125.00> EO-1: Target(tgt-396) window enabled: 501.7 to 600.0


2026-05-19 20:28:46,119 sats.satellite.EO-1            INFO       <125.00> EO-1: setting timed terminal event at 600.0


...

2026-05-19 20:28:46,125 sats.satellite.EO-2            INFO       <140.00> EO-2: timed termination at 140.0 for broadcast


2026-05-19 20:28:46,125 data.base                      INFO       <140.00> Total reward: {}


2026-05-19 20:28:46,126 comm.communication             INFO       <140.00> Communicating data in 1 direction.


2026-05-19 20:28:46,127 sats.satellite.EO-2            INFO       <140.00> EO-2: Satellite EO-2 requires retasking


2026-05-19 20:28:46,129 gym                            INFO       <140.00> Step reward: {}
